# Water Stress Watch v1.5 — Cooling-Type Classifier

**Purpose:** Predict per-facility cooling type (air / hybrid / evaporative) for the 1,575 v0 facilities, so the v1.5 WUE model can use it as a real feature instead of the v1.0 'unknown' proxy.

**Author:** Hiroaki Oshima (interactive execution on Colab Pro)
**Generated by:** `src/build_cooling_classifier.py` (training set) + `src/build_v1_features.py` (inference features)
**Locked decisions (Week 6):** see `docs/handoff_week_6.md`

## What this notebook does
1. Loads the cooling classifier training set (~67 rows: 43 seed from operator disclosures + 24 augmented from public sources).
2. Loads the v1 inference features (1,575 US data centers from v0).
3. Engineers features: lat/lon, wet_bulb_c, sqft, operator_class, climate_zone.
4. **Trains two models:**
   - 4-class XGBoost (air / hybrid / evaporative / water) — the journalism-class output
   - 2-class XGBoost (low_water / high_water) — the actual feature for the v1.5 WUE model
5. **Validates** with 3-fold stratified k-fold (5-fold is too small for the 7 hybrid examples).
6. Predicts cooling type for all 1,575 v0 facilities.
7. Saves the predicted_cooling_type + cooling_class_2 columns to `data/processed/cooling_type_predicted.csv`.
8. Re-runs `src/build_v1_features.py` to add the cooling_type column to `v1_inference_features.csv`.

## What this notebook does NOT do
- Retrain the v1.5 WUE model — that's a separate run after this notebook produces `cooling_type_predicted.csv`
- Optuna hyperparameter sweep on the cooling classifier (default params are good enough for 67 rows)
- Sub-basin (HUC-8) stress overlay (v2)

## Run order
Run cells 1-12 sequentially. Cell 12 (saving) is the gating step. After that, the next notebook (v1.5 WUE training) reads the output.

## Why 2-class and 4-class
The 4-class problem is too noisy at 67 rows (the 7 hybrid examples are too few for stratified 5-fold). The 2-class reframe (low_water = air+hybrid, high_water = evaporative) gets ~73% accuracy vs ~55% for 4-class. The 2-class output is what feeds into the WUE model; the 4-class output is reported for journalism transparency. See the Week 6 summary for the full diagnostic.

In [ ]:
# Cell 2: pip install
# Colab Pro A100 runtime has most of these.
!pip install --quiet \
    xgboost>=2.0.0 \
    scikit-learn>=1.4.0 \
    pandas>=2.0.0 \
    numpy>=1.24.0
print("pip install done")

In [ ]:
# Cell 3: Imports + constants
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

COOLING_CLASSES_4 = ["air", "hybrid", "evaporative"]  # 4-class taxonomy (water class dropped, no training data)
COOLING_CLASSES_2 = ["low_water", "high_water"]       # 2-class reframe
print("imports ok")

In [ ]:
# Cell 4: Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/water_stress_watch_v1")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted at {DRIVE_DIR}")

In [ ]:
# Cell 5: Load the cooling classifier training set
TRAINING_CSV = DRIVE_DIR / "cooling_classifier_training_set.csv"
if not TRAINING_CSV.exists():
    TRAINING_CSV = Path("cooling_classifier_training_set.csv")

train_df = pd.read_csv(TRAINING_CSV)
print(f"Loaded {len(train_df)} training rows from {TRAINING_CSV}")
print(f"  cooling_type: {train_df['cooling_type'].value_counts().to_dict()}")
print(f"  providers: {train_df['provider'].value_counts().to_dict()}")
print(f"  operator_class: {train_df['operator_class'].value_counts().to_dict()}")

In [ ]:
# Cell 6: Load the v1 inference features (1,575 facilities to score)
INFERENCE_CSV = DRIVE_DIR / "v1_inference_features.csv"
if not INFERENCE_CSV.exists():
    INFERENCE_CSV = Path("v1_inference_features.csv")

inference_df = pd.read_csv(INFERENCE_CSV)
print(f"Loaded {len(inference_df)} inference rows from {INFERENCE_CSV}")
print(f"  columns: {list(inference_df.columns)}")

In [ ]:
# Cell 7: Feature engineering
# Features (same for 4-class and 2-class):
#   continuous: latitude, longitude, wet_bulb_c, sqft_total_space, sqft_colocation_space
#   categorical (one-hot): operator_class, climate_zone

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build the feature matrix for the cooling classifier."""
    X = pd.DataFrame({
        "latitude":             df["latitude"],
        "longitude":            df["longitude"],
        "wet_bulb_c":           df["wet_bulb_c"],
        "sqft_total_space":     df["sqft_total_space"].fillna(0) if "sqft_total_space" in df.columns else 0,
        "sqft_colocation_space": df["sqft_colocation_space"].fillna(0) if "sqft_colocation_space" in df.columns else 0,
    })
    # One-hot operator_class
    if "operator_class" in df.columns:
        op = pd.get_dummies(df["operator_class"], prefix="op")
        X = pd.concat([X, op], axis=1)
    # One-hot climate_zone (coarse 4-bin from wet_bulb_c)
    def climate_zone(wb):
        if pd.isna(wb): return "unknown"
        if wb < 7:  return "cold"
        if wb < 13: return "mild"
        if wb < 17: return "warm"
        return "hot"
    cz = pd.get_dummies(df["wet_bulb_c"].apply(climate_zone), prefix="cz")
    X = pd.concat([X, cz], axis=1)
    return X

X_train = build_features(train_df)
print(f"Training feature matrix: {X_train.shape}")
print(f"  columns: {list(X_train.columns)}")

# For inference, we need the same columns. Some inference facilities may
# have operator_class values not seen in training (e.g. 'unknown', 'cable_telecom').
# We'll reindex the inference matrix to match the training columns.
X_inference = build_features(inference_df)
X_inference = X_inference.reindex(columns=X_train.columns, fill_value=0)
print(f"\nInference feature matrix: {X_inference.shape}")
print(f"  same columns as training: {list(X_inference.columns) == list(X_train.columns)}")

In [ ]:
# Cell 8: 4-class XGBoost + 3-fold stratified CV
# Note: 3-fold (not 5-fold) because the hybrid class has only 7 examples.

le4 = LabelEncoder()
y_train_4 = le4.fit_transform(train_df["cooling_type"].values)
print(f"4-class labels: {dict(zip(le4.classes_, range(len(le4.classes_))))}")
print(f"  class counts: {dict(zip(*np.unique(y_train_4, return_counts=True)))}")

xgb_params_4 = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
}

print("\n=== 4-class: 3-fold stratified CV ===")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
fold_acc_4 = []
all_y_true_4 = []
all_y_pred_4 = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train_4), 1):
    m = xgb.XGBClassifier(**xgb_params_4)
    m.fit(X_train.iloc[tr_idx], y_train_4[tr_idx], verbose=False)
    pred = m.predict(X_train.iloc[val_idx])
    acc = accuracy_score(y_train_4[val_idx], pred)
    fold_acc_4.append(acc)
    all_y_true_4.extend(y_train_4[val_idx])
    all_y_pred_4.extend(pred)
    print(f"  fold {fold}: acc={acc:.3f}")
print(f"\n4-class mean accuracy: {np.mean(fold_acc_4):.3f}")
print("\n=== 4-class classification report (all folds) ===")
print(classification_report(all_y_true_4, all_y_pred_4, target_names=le4.classes_, zero_division=0))
print("=== 4-class confusion matrix (rows=true, cols=pred) ===")
cm = confusion_matrix(all_y_true_4, all_y_pred_4)
print(pd.DataFrame(cm, index=le4.classes_, columns=le4.classes_))

In [ ]:
# Cell 9: 2-class XGBoost (the real signal) + 3-fold stratified CV
# low_water = air + hybrid (WUE 0.2-0.5, dominated by humidification)
# high_water = evaporative (WUE 0.5-1.8, cooling tower water use)

train_df_2 = train_df.copy()
train_df_2["cooling_class_2"] = train_df_2["cooling_type"].map({
    "air": "low_water",
    "hybrid": "low_water",
    "evaporative": "high_water",
})
y_train_2 = (train_df_2["cooling_class_2"] == "high_water").astype(int).values
print(f"2-class target: {y_train_2.sum()} high_water / {len(y_train_2)} ({y_train_2.mean():.1%})")

xgb_params_2 = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "tree_method": "hist",
    "eval_metric": "logloss",
}

print("\n=== 2-class: 3-fold stratified CV ===")
skf2 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
fold_acc_2 = []
all_y_true_2 = []
all_y_pred_2 = []
for fold, (tr_idx, val_idx) in enumerate(skf2.split(X_train, y_train_2), 1):
    m = xgb.XGBClassifier(**xgb_params_2)
    m.fit(X_train.iloc[tr_idx], y_train_2[tr_idx], verbose=False)
    pred = m.predict(X_train.iloc[val_idx])
    acc = accuracy_score(y_train_2[val_idx], pred)
    fold_acc_2.append(acc)
    all_y_true_2.extend(y_train_2[val_idx])
    all_y_pred_2.extend(pred)
    print(f"  fold {fold}: acc={acc:.3f}")
print(f"\n2-class mean accuracy: {np.mean(fold_acc_2):.3f}")
print("\n=== 2-class classification report (all folds) ===")
print(classification_report(all_y_true_2, all_y_pred_2, target_names=["low_water", "high_water"]))
print("=== 2-class confusion matrix (rows=true, cols=pred) ===")
cm2 = confusion_matrix(all_y_true_2, all_y_pred_2)
print(pd.DataFrame(cm2, index=["low_water", "high_water"], columns=["low_water", "high_water"]))

# Note for Hiroaki: 70% is a target, not a guarantee. With 67 labels the
# noise floor is around 70%. If the 2-class accuracy comes out at 70-80%,
# that's the honest answer. Below 70% means the climate signal is not strong
# enough — we'd need either more labeled data or per-facility press releases
# naming the cooling vendor (out of scope for v1.5).

In [ ]:
# Cell 10: Full-data fit on both models
model_4 = xgb.XGBClassifier(**xgb_params_4)
model_4.fit(X_train, y_train_4, verbose=False)
print(f"4-class model trained on {len(X_train)} rows")
print("\n4-class feature importance:")
imp_4 = pd.Series(model_4.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp_4.to_string())

model_2 = xgb.XGBClassifier(**xgb_params_2)
model_2.fit(X_train, y_train_2, verbose=False)
print(f"\n2-class model trained on {len(X_train)} rows")
print("\n2-class feature importance:")
imp_2 = pd.Series(model_2.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp_2.to_string())

In [ ]:
# Cell 11: Predict cooling type for all 1,575 v0 facilities
pred_4_idx = model_4.predict(X_inference)
pred_4_proba = model_4.predict_proba(X_inference)
pred_2 = model_2.predict(X_inference)
pred_2_proba = model_2.predict_proba(X_inference)

inference_df["predicted_cooling_type"] = le4.inverse_transform(pred_4_idx)
inference_df["cooling_class_2"] = ["high_water" if p == 1 else "low_water" for p in pred_2]

# Probability columns for transparency
for i, cls in enumerate(le4.classes_):
    inference_df[f"prob_{cls}"] = pred_4_proba[:, i]
inference_df["prob_high_water"] = pred_2_proba[:, 1]

print("=== Predicted cooling type distribution (1,575 facilities) ===")
print(f"  4-class: {inference_df['predicted_cooling_type'].value_counts().to_dict()}")
print(f"  2-class: {inference_df['cooling_class_2'].value_counts().to_dict()}")

print("\n=== By operator_class (top 3) ===")
ct = pd.crosstab(inference_df["operator_class"], inference_df["cooling_class_2"], normalize="index")
print(ct.round(2))

print("\n=== By state (top 10 by facility count) ===")
top_states = inference_df["state"].value_counts().head(10).index.tolist()
ct_state = pd.crosstab(
    inference_df[inference_df["state"].isin(top_states)]["state"],
    inference_df[inference_df["state"].isin(top_states)]["cooling_class_2"],
    normalize="index",
)
print(ct_state.round(2))

In [ ]:
# Cell 12: Save model + predictions to Drive
# 12a: 4-class model artifact
model_4_path = DRIVE_DIR / "cooling_classifier_4class.pkl"
joblib.dump({"model": model_4, "label_encoder": le4}, model_4_path)
print(f"4-class model saved to {model_4_path}")

# 12b: 2-class model artifact
model_2_path = DRIVE_DIR / "cooling_classifier_2class.pkl"
joblib.dump({"model": model_2}, model_2_path)
print(f"2-class model saved to {model_2_path}")

# 12c: predicted cooling type per facility (the journalism output)
preds_path = DRIVE_DIR / "cooling_type_predicted.csv"
out_cols = [
    "dc_id", "name", "provider", "state", "latitude", "longitude",
    "operator_class", "wet_bulb_c",
    "predicted_cooling_type", "cooling_class_2",
    "prob_air", "prob_hybrid", "prob_evaporative",
    "prob_high_water",
]
inference_df[out_cols].to_csv(preds_path, index=False)
print(f"\nPredictions saved to {preds_path}")
print(f"  rows: {len(inference_df)}")
print(f"  columns: {out_cols}")

# 12d: validation metrics summary (for the auto-summary cell)
import json
metrics_path = DRIVE_DIR / "cooling_classifier_metrics.json"
metrics = {
    "n_train": len(X_train),
    "n_inference": len(X_inference),
    "4class": {
        "cv_mean_accuracy": float(np.mean(fold_acc_4)),
        "cv_fold_accuracies": [float(a) for a in fold_acc_4],
        "class_labels": list(le4.classes_),
    },
    "2class": {
        "cv_mean_accuracy": float(np.mean(fold_acc_2)),
        "cv_fold_accuracies": [float(a) for a in fold_acc_2],
        "class_labels": ["low_water", "high_water"],
    },
    "predicted_4class_distribution": inference_df["predicted_cooling_type"].value_counts().to_dict(),
    "predicted_2class_distribution": inference_df["cooling_class_2"].value_counts().to_dict(),
}
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to {metrics_path}")

print("\nAfter the run, download these three files to:")
print("  /root/project/datacenter_water_stress/models/cooling_classifier_4class.pkl")
print("  /root/project/datacenter_water_stress/models/cooling_classifier_2class.pkl")
print("  /root/project/datacenter_water_stress/data/processed/cooling_type_predicted.csv")
print("\nThen re-run src/build_v1_features.py to fold cooling_type into v1_inference_features.csv.")
print("Then re-run notebooks/04_ml_training.ipynb with v1.5 settings (cooling_type as a real feature).")

In [ ]:
# Cell 13: Auto-generated run summary (re-runs every time)
import json
from datetime import datetime

metrics = json.load(open(DRIVE_DIR / "cooling_classifier_metrics.json"))

print("=" * 70)
print("  v1.5 Cooling-Type Classifier — Run Summary")
print("=" * 70)
print(f"  Run date:        {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Training set:    {metrics['n_train']} rows")
print(f"  Inference set:   {metrics['n_inference']} rows (1,575 v0 facilities)")
print()
print("  --- 4-class model (air / hybrid / evaporative) ---")
print(f"  CV mean accuracy: {metrics['4class']['cv_mean_accuracy']:.3f}")
print(f"  Fold accuracies:  {metrics['4class']['cv_fold_accuracies']}")
print(f"  Classes:          {metrics['4class']['class_labels']}")
print()
print("  --- 2-class model (low_water / high_water) ---")
print(f"  CV mean accuracy: {metrics['2class']['cv_mean_accuracy']:.3f}")
print(f"  Fold accuracies:  {metrics['2class']['cv_fold_accuracies']}")
print(f"  Classes:          {metrics['2class']['class_labels']}")
print()
print("  --- Predicted distribution on 1,575 v0 facilities ---")
for k, v in metrics["predicted_4class_distribution"].items():
    print(f"    {k:12s}: {v}")
print()
for k, v in metrics["predicted_2class_distribution"].items():
    pct = 100 * v / metrics["n_inference"]
    print(f"    {k:12s}: {v} ({pct:.1f}%)")
print()
print("  Next step: re-run src/build_v1_features.py, then notebooks/04_ml_training.ipynb")
print("  with v1.5 settings (cooling_type as a real categorical feature).")

# v1.5 Cooling Classifier — Discovery & Verdict

**Run date:** 2026-07-04 (Week 6, Task 6.1: cooling-type classifier)
**Author:** Hiroaki Oshima with assistance from Hermes Agent

## Why this task

The v1.0 WUE model collapsed to a 1.5-feature model (85% of feature importance on the binary `cooling_type` flag) because v0 didn't capture per-facility cooling type. The v1.5 fix: predict cooling type first, then use the prediction as a real feature in the v1.5 WUE model.

## What got built

1. **`src/build_cooling_classifier.py`** — assembles the 67-row labeled training set (43 disclosed + 24 augmented from public sources)
2. **`data/processed/cooling_classifier_augmented_labels.csv`** — the 24 augmented rows (Meta 2024 per-site hybrid + Google 2024 per-site + Equinix/Digital Realty industry defaults)
3. **`data/processed/cooling_classifier_training_set.csv`** — the combined 67-row training set with wet_bulb_c and sqft attached via spatial nearest-neighbor
4. **`src/build_v1_features.py` (updated)** — now reads `cooling_type_predicted.csv` and adds the `cooling_type` column to the v1 inference features
5. **`notebooks/06_cooling_classifier.ipynb`** — the training notebook (this file)

## What the dry-run found (Week 6 Hermes session)

The 4-class problem is too noisy at 67 rows. The 7 hybrid examples are too few for stratified 5-fold, and the air/hybrid boundary is subtle. The 2-class reframe (low_water = air+hybrid, high_water = evaporative) gets ~73% accuracy vs ~55% for 4-class. A simple rules-based baseline ("hyperscaler = high_water, colo = low_water") already gets 70%.

**The honest verdict:** with 67 labels, the noise floor is ~70%. The cooling type is determined by engineering choice at design time, not inferable from climate + operator class alone. To do much better would require per-facility press releases naming the cooling vendor or mechanical spec sheets (out of scope for v1.5).

## What this enables for the v1.5 WUE model

The v1.5 WUE model uses `cooling_class_2` (low_water / high_water) as a real categorical feature instead of the v1.0 'unknown' proxy. The expected improvement:
- v1.0 LOO Meta R² = -717 (the collapse)
- v1.5 LOO Meta R² should be > 0 (target)
- v1.5 5-fold R² should be higher than v1.0's 0.651 (target: 0.70+)

## What this does NOT enable

- Per-facility cooling type with high confidence (the 4-class model is ~55% accurate)
- Disambiguating air from hybrid (the boundary is subtle; only 7 hybrid examples)
- Cooling type for facilities with operator_class='unknown' (the model degrades to the prior)

## Locked decisions (Week 6)

- 2-class reframe is the right level of granularity for the WUE model (air vs evaporative is the physically meaningful axis)
- 4-class output is reported for journalism transparency, but with a caveat about the air/hybrid boundary
- 3-fold CV (not 5-fold) because the 7 hybrid examples are too few for 5-fold
- Climate zone (cold/mild/warm/hot from wet_bulb_c bins) is a useful synthetic feature; documented in Cell 7